In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
!pip install -U ipywidgets

In [ ]:
# ============================================================
# FINAL — Healthcare Management System (patched)
# - Patient Records (50 dummy)
# - Add / View / Search / Update / Delete
# - Appointments
# - AI Recommender (Professional)
# - Billing & Payment (Base + Disease costs + Manual override + Half/Full)
# Patches applied:
# - billing_amount_display uses HTML (styled)
# - calculate_amount_for recalculates base fees and disease cost
# - billing_calculate_clicked updates HTML widget value
# - billing_calc_btn click binding simplified
# ============================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, date
import random, re, time

random.seed(42)

# ---------------------------
# UI CSS
# ---------------------------
css = """
<style>
body { font-family: 'Segoe UI', Arial, sans-serif; }
.header {
  color:#fff; background: linear-gradient(90deg,#0f172a,#0ea5e9);
  padding:12px;border-radius:10px;text-align:center;margin-bottom:12px;
  box-shadow:0 8px 24px rgba(14,165,233,0.12);
}
.card{border:1px solid #eef2ff;border-radius:10px;padding:12px;margin-bottom:10px;background:#fff}
.dataframe-div{background:#fbfcff;padding:8px;border-radius:10px;box-shadow:0 2px 10px rgba(2,6,23,0.03)}
.small{color:#6b7280;font-size:12px}
.info{background:#eef2ff;padding:8px;border-radius:6px;margin-bottom:8px}
</style>
"""
display(HTML(css))
display(HTML("<div class='header'>🏥 Final Healthcare Management System — Complete (Patched)</div>"))

# ---------------------------
# Gender detection helpers
# ---------------------------
masculine_names = {"aarav","vivaan","aditya","arjun","krishna","aryan","vihaan","kabir","rohan","ishaan","karan","aman","yash","pranav","vivek","ravi","rohit","sumit","rahul"}
feminine_names = {"riya","sanya","ananya","diya","neha","shruti","sneha","pooja","priya","aisha","meera","simran","kavya","nisha","kriti","anika","sakshi"}

def detect_gender_from_name(name: str) -> str:
    if not name or not isinstance(name, str):
        return "Other"
    first = name.strip().split()[0].lower()
    first_clean = re.sub(r'[^a-z]','', first)
    if not first_clean: return "Other"
    if first_clean in masculine_names: return "Male"
    if first_clean in feminine_names: return "Female"
    if first_clean[-1] in set("aiey"): return "Female"
    if first_clean[-1] in set("bcdfghjklmnpqrstvwxyz"): return "Male"
    return "Other"

# ---------------------------
# Disease cost table (example professional mapping)
# ---------------------------
BASE_CHARGES = {
    "Consultation": 500,
    "Registration": 200,
    "Service": 100
}
BASE_TOTAL = sum(BASE_CHARGES.values())

DISEASE_COSTS = {
    "Fever": 300,
    "Flu": 400,
    "Infection": 800,
    "Diabetes": 1500,
    "Hypertension": 1200,
    "Fracture": 3500,
    "Surgery": 20000,
    "Cardiac Issue": 15000,
    "Asthma": 900,
    "Allergy": 350,
    "Cold": 300,
    "Gastritis": 500,
    "Skin Infection": 600,
    "UTI": 700,
    "Anemia": 600,
    "Depression": 1000,
    "Thyroid": 1200,
    "Sprain": 400,
    "Ear Infection": 350,
    "Eye Infection": 400
}

# ---------------------------
# AI Recommender KB — Professional (Option 3)
# ---------------------------
AI_KB = {
    "Fever": {
        "summary": "Symptomatic treatment + check for infection or viral cause.",
        "medicines": [
            {"name":"Paracetamol","dose":"500 mg","frequency":"every 4-6 hours (max 3 g/day)","duration":"as needed 3-5 days","notes":"Antipyretic for fever."},
            {"name":"ORS","dose":"200-300 ml","frequency":"every few hours","duration":"until hydration restored","notes":"Maintain hydration."}
        ],
        "tests":["CBC","CRP","Blood Culture (if sepsis suspected)"],
        "emergency_triggers":["Persistent high fever > 40°C","Difficulty breathing","Confusion or seizures","Signs of dehydration"],
        "advice":"Rest, hydrate, monitor fever; seek urgent care if emergency signs appear."
    },
    "Asthma": {
        "summary": "Reliever + controller approach depending on severity.",
        "medicines":[
            {"name":"Salbutamol inhaler (SABA)","dose":"2 puffs","frequency":"as needed","duration":"PRN","notes":"Reliever inhaler."},
            {"name":"Budesonide inhaler (ICS)","dose":"200 mcg","frequency":"1-2 puffs daily","duration":"long-term","notes":"Controller therapy."}
        ],
        "tests":["Peak flow measurement","Spirometry (if available)"],
        "emergency_triggers":["Severe breathlessness at rest","Unable to speak in full sentences","Blue lips/fingertips"],
        "advice":"Avoid triggers; use inhaler technique properly; action plan for attacks."
    },
    "Diabetes": {
        "summary":"Glycemic control with medications, diet, and monitoring.",
        "medicines":[
            {"name":"Metformin","dose":"500 mg","frequency":"twice daily","duration":"long-term","notes":"First-line for type 2 diabetes."}
        ],
        "tests":["Fasting blood glucose","HbA1c","Urea & electrolyte (KFT)"],
        "emergency_triggers":["Very high blood glucose (>300 mg/dL)","Persistent vomiting/dehydration","Altered consciousness"],
        "advice":"Dietary control, regular exercise, monitor glucose at home, follow-up with endocrinologist."
    },
    "Infection": {
        "summary":"Antibiotic therapy guided by suspected source and cultures.",
        "medicines":[
            {"name":"Amoxicillin-clavulanate","dose":"625 mg","frequency":"three times daily","duration":"5-7 days","notes":"Broad-spectrum oral antibiotic (if indicated)."}
        ],
        "tests":["CBC","Culture & sensitivity (blood/urine/wound)"],
        "emergency_triggers":["High fever, hypotension, altered mentation","Signs of sepsis"],
        "advice":"Complete full antibiotic course if prescribed; return if worsening."
    },
    "Default": {
        "summary":"Symptomatic care; consult a physician for tailored therapy.",
        "medicines":[{"name":"Paracetamol","dose":"500 mg","frequency":"as needed","duration":"short term","notes":"For pain/fever."}],
        "tests":["As advised by physician"],
        "emergency_triggers":["Severe or worsening symptoms"],
        "advice":"See doctor for a detailed plan."
    }
}

# ---------------------------
# Demo dataset: 50 patients
# ---------------------------
first_names = ["Aarav","Vivaan","Aditya","Arjun","Krishna","Shaurya","Aryan","Vihaan","Kabir","Rohan",
               "Ishaan","Karan","Sai","Ayaan","Reyansh","Dev","Mihir","Harsh","Tanmay","Nikhil",
               "Riya","Sanya","Ananya","Diya","Neha","Shruti","Sneha","Pooja","Priya","Aisha",
               "Meera","Simran","Kavya","Ira","Nisha","Shreya","Ritika","Kriti","Anika","Sakshi",
               "Aman","Ritvik","Advik","Yash","Angad","Lokesh","Siddhant","Manav","Pranav","Vivek"]

doctors = ["Dr. Mehta","Dr. Sharma","Dr. Verma","Dr. Kapoor","Dr. Rao","Dr. Singh","Dr. Iyer"]
tests_catalog = ["Blood","X-Ray","ECG","MRI","Ultrasound","CBP","LFT","KFT"]
conditions = ["Stable","Recovering","Critical"]

def gen_patients(n=50, start=100):
    rows=[]
    for i in range(n):
        name = first_names[i % len(first_names)]
        pid = f"P{start+i}"
        age = random.randint(3,80)
        gender = detect_gender_from_name(name)
        disease = random.choice(list(DISEASE_COSTS.keys()))
        doc = random.choice(doctors)
        admission = date.today().strftime("%Y-%m-%d")
        tests = random.sample(tests_catalog, k=random.randint(0,2))
        bill_extra = DISEASE_COSTS.get(disease, 500)
        bill = BASE_TOTAL + bill_extra
        paid = round(random.choice([0.0, bill*0.25, bill*0.5, bill]),2)
        cond = random.choices(conditions, weights=[0.75,0.2,0.05])[0]
        rows.append({
            "PatientID":pid,"Name":name,"Age":age,"Gender":gender,"Disease":disease,
            "DoctorSuggested":doc,"AdmissionDate":admission,"Tests":tests,
            "BillAmount":float(bill),"PaidAmount":float(paid),"CurrentCondition":cond
        })
    return pd.DataFrame(rows)

patients = gen_patients(50,100)

# Transactions & appointments
transactions = []
appointments = []

# ---------------------------
# Utilities & styling
# ---------------------------
def next_patient_id():
    if patients.empty:
        return "P100"
    nums = patients["PatientID"].str.replace("P","",regex=False).astype(int)
    return f"P{nums.max()+1}"

def style_table(df, caption=None, width=1100):
    styles=[dict(selector="th", props=[("background","#0ea5e9"),("color","white"),("padding","8px")]),
            dict(selector="td", props=[("padding","6px")]),
            dict(selector="table", props=[("border-collapse","collapse"),("border","1px solid #e6eef8"),("font-family","Segoe UI, Arial"),("font-size","13px"),("width",f"{width}px")])]
    st = df.style.set_table_styles(styles).hide(axis="index")
    if caption: st = st.set_caption(caption)
    return st

def log_transaction(pid, name, amt, ttype):
    # called after PaidAmount updated
    if pid not in patients["PatientID"].tolist(): return
    row = patients.loc[patients["PatientID"]==pid].iloc[0]
    balance_after = float(row["BillAmount"] - row["PaidAmount"])
    transactions.append({
        "Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "PatientID":pid,"Name":name,"Type":ttype,"Amount":float(amt),"BalanceAfter":round(balance_after,2)
    })

# ---------------------------
# Widgets & outputs
# ---------------------------
out_records = widgets.Output()
out_add = widgets.Output()
out_view = widgets.Output()
out_search = widgets.Output()
out_update = widgets.Output()
out_billing = widgets.Output()
out_transactions = widgets.Output()
out_appointments = widgets.Output()
out_ai = widgets.Output()

# Common controls
pid_dropdown = widgets.Dropdown(description="Patient:", layout=widgets.Layout(width="380px"))
apply_filters_btn = widgets.Button(description="Apply Filters", button_style="info")
search_text = widgets.Text(description="Search:", placeholder="name or ID", layout=widgets.Layout(width="420px"))
disease_filter = widgets.Dropdown(options=["All"]+sorted(list(DISEASE_COSTS.keys())), value="All", description="Disease:")
condition_filter = widgets.Dropdown(options=["All"]+conditions, value="All", description="Condition:")
gender_filter = widgets.Dropdown(options=["All","Male","Female","Other"], value="All", description="Gender:")

# Add patient controls
add_name = widgets.Text(description="Name:", layout=widgets.Layout(width="360px"))
add_age = widgets.BoundedIntText(description="Age:", min=0, max=120, value=25, layout=widgets.Layout(width="140px"))
add_gender = widgets.Dropdown(description="Gender:", options=["Auto","Male","Female","Other"], value="Auto", layout=widgets.Layout(width="180px"))
add_disease = widgets.Dropdown(description="Disease:", options=sorted(list(DISEASE_COSTS.keys())), layout=widgets.Layout(width="260px"))
add_doctor = widgets.Dropdown(description="Doctor:", options=doctors, layout=widgets.Layout(width="260px"))
add_tests = widgets.SelectMultiple(options=tests_catalog, description="Tests:")
add_base_fee = widgets.BoundedIntText(description="Base Fee:", min=0, value=BASE_TOTAL, layout=widgets.Layout(width="160px"))
add_btn = widgets.Button(description="Add Patient", button_style="primary", icon="plus")

# Update/Delete controls
ud_select = widgets.Dropdown(description="Select Patient:", layout=widgets.Layout(width="360px"))
ud_name = widgets.Text(description="Name:", layout=widgets.Layout(width="360px"))
ud_age = widgets.BoundedIntText(description="Age:", min=0, max=120, layout=widgets.Layout(width="120px"))
ud_gender = widgets.Dropdown(description="Gender:", options=["Male","Female","Other"], layout=widgets.Layout(width="140px"))
ud_disease = widgets.Text(description="Disease:", layout=widgets.Layout(width="260px"))
ud_doctor = widgets.Text(description="Doctor:", layout=widgets.Layout(width="260px"))
ud_bill = widgets.BoundedFloatText(description="Bill ₹:", min=0, value=0, layout=widgets.Layout(width="140px"))
ud_paid = widgets.BoundedFloatText(description="Paid ₹:", min=0, value=0, layout=widgets.Layout(width="140px"))
ud_tests = widgets.Text(description="Tests (comma):", layout=widgets.Layout(width="360px"))
ud_condition = widgets.Dropdown(description="Condition:", options=conditions, layout=widgets.Layout(width="160px"))
ud_update = widgets.Button(description="Update", button_style="success")
ud_delete = widgets.Button(description="Delete", button_style="danger")
ud_refresh = widgets.Button(description="Refresh", button_style="info")

# Billing controls (PATCHED: use HTML for styled amount display)
billing_pid = widgets.Dropdown(description="Patient:", layout=widgets.Layout(width="360px"))
billing_amount_display = widgets.HTML(
    "<div style='font-size:16px;font-weight:600;color:#0f172a;padding:6px 12px;background:#f0f9ff;border-radius:8px;border:1px solid #bae6fd;width:220px;'>Amount: ₹0.00</div>"
)
billing_custom_override = widgets.BoundedFloatText(description="Manual override ₹:", min=0, value=0, layout=widgets.Layout(width="220px"))
billing_calc_btn = widgets.Button(description="Calculate Amount", button_style="info")
billing_half_btn = widgets.Button(description="Pay Half", button_style="warning")
billing_full_btn = widgets.Button(description="Pay Full", button_style="success")
billing_msg = widgets.HTML()

# Transactions view button
view_tx_btn = widgets.Button(description="Refresh Transactions", button_style="info")

# Appointments controls
appoint_patient = widgets.Dropdown(description="Patient:", layout=widgets.Layout(width="260px"))
appoint_doctor = widgets.Dropdown(description="Doctor:", options=doctors, layout=widgets.Layout(width="260px"))
appoint_date = widgets.DatePicker(description="Date:")
appoint_time = widgets.Text(description="Time:", placeholder="HH:MM")
appoint_reason = widgets.Text(description="Reason:", layout=widgets.Layout(width="300px"))
book_btn = widgets.Button(description="Book Appointment", button_style="primary")

# AI recommender
ai_disease = widgets.Dropdown(description="Disease:", options=sorted(list(AI_KB.keys())), layout=widgets.Layout(width="360px"))
ai_btn = widgets.Button(description="Get Professional Recommendation", button_style="info")
ai_clear = widgets.Button(description="Clear", button_style="warning")

# ---------------------------
# Refresh helpers
# ---------------------------
def refresh_dropdowns():
    pids = patients["PatientID"].tolist()
    if pids:
        pid_dropdown.options = pids
        billing_pid.options = pids
        ud_select.options = pids
        appoint_patient.options = pids
        appoint_patient.value = pids[0] if appoint_patient.value not in pids else appoint_patient.value
    else:
        pid_dropdown.options = []
        billing_pid.options = []
        ud_select.options = []
        appoint_patient.options = []

def refresh_filters():
    disease_filter.options = ["All"] + sorted(patients["Disease"].dropna().unique().tolist())
    # keep previous selections safe
    if condition_filter.value not in ["All"]+conditions:
        condition_filter.value = "All"

# ---------------------------
# Render Patient Records tab (first tab)
# ---------------------------
def render_records():
    with out_records:
        clear_output()
        if patients.empty:
            display(HTML("<b>No records.</b>"))
            return
        df = patients.copy()
        df_display = df.copy()
        df_display["Tests"] = df_display["Tests"].apply(lambda x:", ".join(x) if isinstance(x,(list,tuple)) else str(x))
        display(HTML("<div class='card'><b>Patient Records (50 Dummy Patients)</b></div>"))
        display(HTML("<div class='dataframe-div'>"))
        display(style_table(df_display[["PatientID","Name","Age","Gender","Disease","DoctorSuggested","AdmissionDate","Tests","BillAmount","PaidAmount","CurrentCondition"]].reset_index(drop=True), caption="Patient Records"))
        display(HTML("</div>"))

# ---------------------------
# View / Filter patients
# ---------------------------
def apply_filters(_=None):
    with out_view:
        clear_output()
        df = patients.copy()
        q = (search_text.value or "").strip().lower()
        if q:
            df = df[df["Name"].fillna("").str.lower().str.contains(q) | df["PatientID"].fillna("").str.lower().str.contains(q)]
        if disease_filter.value != "All":
            df = df[df["Disease"]==disease_filter.value]
        if condition_filter.value != "All":
            df = df[df["CurrentCondition"]==condition_filter.value]
        if gender_filter.value != "All":
            df = df[df["Gender"]==gender_filter.value]
        if df.empty:
            display(HTML("<b>No patients match the filters.</b>"))
            return
        df2 = df.copy()
        df2["Tests"] = df2["Tests"].apply(lambda x:", ".join(x) if isinstance(x,(list,tuple)) else str(x))
        display(style_table(df2[["PatientID","Name","Age","Gender","Disease","DoctorSuggested","AdmissionDate","Tests","BillAmount","PaidAmount","CurrentCondition"]].reset_index(drop=True), caption="Filtered Patients"))

apply_filters_btn.on_click(apply_filters)

# ---------------------------
# Add patient logic (global used)
# ---------------------------
def add_patient_clicked(_):
    global patients
    name = (add_name.value or "").strip()
    if not name:
        with out_add:
            clear_output(); display(HTML("<b style='color:red'>Name required.</b>")); return
    pid = next_patient_id()
    age = int(add_age.value)
    gval = add_gender.value
    gender_val = detect_gender_from_name(name) if gval=="Auto" else gval
    disease = add_disease.value
    doc = add_doctor.value
    tests = list(add_tests.value)
    bill = int(add_base_fee.value) + DISEASE_COSTS.get(disease,0)
    paid = 0.0
    row = {
        "PatientID":pid,"Name":name,"Age":age,"Gender":gender_val,"Disease":disease,
        "DoctorSuggested":doc,"AdmissionDate":date.today().strftime("%Y-%m-%d"),
        "Tests":tests,"BillAmount":float(bill),"PaidAmount":float(paid),"CurrentCondition":"Stable"
    }
    patients = pd.concat([patients, pd.DataFrame([row])], ignore_index=True)
    with out_add:
        clear_output(); display(HTML(f"<b style='color:green'>Added patient {name} ({pid})</b>"))
    refresh_all()

add_btn.on_click(add_patient_clicked)

# ---------------------------
# Update/Delete logic
# ---------------------------
def ud_load(change=None):
    with out_update:
        clear_output()
        pid = ud_select.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b>No patient selected.</b>")); return
        row = patients[patients["PatientID"]==pid].iloc[0]
        ud_name.value = row["Name"]
        ud_age.value = int(row["Age"])
        ud_gender.value = row["Gender"] if row["Gender"] in ["Male","Female","Other"] else "Other"
        ud_disease.value = row["Disease"]
        ud_doctor.value = row["DoctorSuggested"]
        ud_bill.value = float(row["BillAmount"])
        ud_paid.value = float(row["PaidAmount"])
        ud_tests.value = ", ".join(row["Tests"]) if isinstance(row["Tests"],(list,tuple)) else str(row["Tests"])
        ud_condition.value = row["CurrentCondition"]
        display(HTML(f"<div class='info'>Loaded {row['Name']} ({pid})</div>"))

def ud_update_clicked(_):
    global patients
    with out_update:
        clear_output()
        pid = ud_select.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b style='color:red'>Select valid patient.</b>")); return
        idx = patients.index[patients["PatientID"]==pid][0]
        patients.at[idx,"Name"] = ud_name.value.strip()
        patients.at[idx,"Age"] = int(ud_age.value)
        patients.at[idx,"Gender"] = ud_gender.value
        patients.at[idx,"Disease"] = ud_disease.value.strip()
        patients.at[idx,"DoctorSuggested"] = ud_doctor.value.strip()
        patients.at[idx,"BillAmount"] = float(ud_bill.value)
        patients.at[idx,"PaidAmount"] = float(ud_paid.value)
        tests_list = [t.strip() for t in ud_tests.value.split(",") if t.strip()]
        patients.at[idx,"Tests"] = tests_list
        patients.at[idx,"CurrentCondition"] = ud_condition.value
        display(HTML(f"<b style='color:green'>Patient {pid} updated.</b>"))
    refresh_all()

def ud_delete_clicked(_):
    global patients
    with out_update:
        clear_output()
        pid = ud_select.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b style='color:red'>Select valid patient.</b>")); return
        idx = patients.index[patients["PatientID"]==pid][0]
        name = patients.at[idx,"Name"]
        patients = patients.drop(index=idx).reset_index(drop=True)
        display(HTML(f"<b style='color:orange'>Deleted {name} ({pid}).</b>"))
    refresh_all()

ud_select.observe(ud_load, names="value")
ud_update.on_click(ud_update_clicked)
ud_delete.on_click(ud_delete_clicked)
ud_refresh.on_click(lambda _: refresh_all())

# ---------------------------
# Billing calculation & payments (PATCHED)
# ---------------------------
def calculate_amount_for(pid):
    """Always compute correct bill from base fees + disease charges."""
    if pid not in patients["PatientID"].tolist():
        return 0.0
    row = patients[patients["PatientID"] == pid].iloc[0]
    disease = row["Disease"]
    base_total = sum(BASE_CHARGES.values())  # recalc base from dict
    disease_cost = DISEASE_COSTS.get(disease, 0)
    total = float(base_total + disease_cost)
    return total

def billing_calculate_clicked(_):
    with out_billing:
        clear_output()
        pid = billing_pid.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b style='color:red'>Select patient.</b>")); return
        suggested = calculate_amount_for(pid)
        override = float(billing_custom_override.value or 0)
        final_amount = override if override>0 else suggested
        # Update styled HTML widget
        billing_amount_display.value = (
            f"<div style='font-size:16px;font-weight:600;color:#0f172a;"
            f"padding:6px 12px;background:#f0f9ff;border-radius:8px;"
            f"border:1px solid #bae6fd;width:220px;'>Amount: ₹{final_amount:.2f}</div>"
        )
        display(HTML(f"<div class='info'>Calculated amount for <b>{pid}</b> — ₹{final_amount:.2f}</div>"))

def billing_pay_half(_):
    global patients
    with out_billing:
        clear_output()
        pid = billing_pid.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b style='color:red'>Select patient.</b>")); return
        suggested = calculate_amount_for(pid)
        final_amount = float(billing_custom_override.value) if float(billing_custom_override.value or 0)>0 else suggested
        to_pay = round(final_amount/2,2)
        idx = patients.index[patients["PatientID"]==pid][0]
        paid_before = patients.at[idx,"PaidAmount"]
        payable = max(0.0, min(to_pay, patients.at[idx,"BillAmount"] - paid_before))
        if payable <= 0:
            display(HTML("<b>Nothing to pay (already paid or zero).</b>")); return
        patients.at[idx,"PaidAmount"] = round(paid_before + payable, 2)
        log_transaction(pid, patients.at[idx,"Name"], payable, "Half Payment")
        display(HTML(f"<div style='color:green;font-weight:700'>Amount Paid Successfully! ₹{payable:.2f}</div>"))
    refresh_all()

def billing_pay_full(_):
    global patients
    with out_billing:
        clear_output()
        pid = billing_pid.value
        if not pid or pid not in patients["PatientID"].tolist():
            display(HTML("<b style='color:red'>Select patient.</b>")); return
        suggested = calculate_amount_for(pid)
        final_amount = float(billing_custom_override.value) if float(billing_custom_override.value or 0)>0 else suggested
        idx = patients.index[patients["PatientID"]==pid][0]
        paid_before = patients.at[idx,"PaidAmount"]
        to_pay = max(0.0, min(final_amount - paid_before, final_amount))
        if to_pay <= 0:
            display(HTML("<b>Nothing to pay (already paid or zero).</b>")); return
        patients.at[idx,"PaidAmount"] = round(paid_before + to_pay, 2)
        log_transaction(pid, patients.at[idx,"Name"], to_pay, "Full Payment")
        display(HTML(f"<div style='color:green;font-weight:700'>Amount Paid Successfully! ₹{to_pay:.2f}</div>"))
    refresh_all()

billing_calc_btn.on_click(billing_calculate_clicked)  # simplified binding
billing_half_btn.on_click(billing_pay_half)
billing_full_btn.on_click(billing_pay_full)
view_tx_btn.on_click(lambda _: show_transactions())

# ---------------------------
# Transactions display
# ---------------------------
def show_transactions():
    with out_transactions:
        clear_output()
        if not transactions:
            display(HTML("<b>No transactions.</b>")); return
        df = pd.DataFrame(transactions)
        display(style_table(df, caption="Transactions"))

# ---------------------------
# Appointments booking
# ---------------------------
def book_appointment_clicked(_):
    pid = appoint_patient.value
    doc = appoint_doctor.value
    dt = appoint_date.value
    t = appoint_time.value.strip()
    reason = appoint_reason.value.strip()
    with out_appointments:
        clear_output()
        if not pid or not dt or not t:
            display(HTML("<b style='color:red'>Select patient, date and time.</b>")); return
        ap = {"AppointmentID":f"A{len(appointments)+1:03d}","PatientID":pid,
              "PatientName": patients.loc[patients['PatientID']==pid,'Name'].iloc[0],
              "Doctor":doc,"Date":str(dt),"Time":t,"Reason":reason,"BookedAt":datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
        appointments.append(ap)
        display(HTML(f"<div style='color:green'>Appointment booked ({ap['AppointmentID']})</div>"))
    show_booked()

def show_booked():
    with out_appointments:
        clear_output()
        if not appointments:
            display(HTML("<b>No appointments booked.</b>")); return
        df = pd.DataFrame(appointments)
        display(style_table(df, caption="Booked Appointments"))

book_btn.on_click(book_appointment_clicked)

# ---------------------------
# AI Recommender — professional display
# ---------------------------
def ai_get_reco(_):
    with out_ai:
        clear_output()
        d = ai_disease.value or ""
        if not d:
            display(HTML("<b>Select disease.</b>")); return
        entry = AI_KB.get(d, AI_KB["Default"])
        html = "<div class='card' style='width:800px'>"
        html += f"<h3 style='margin:0'>{d} — Professional Recommendation</h3>"
        html += f"<p><b>Summary:</b> {entry.get('summary','')}</p>"
        html += "<p><b>Medicines:</b></p><ul>"
        for m in entry.get("medicines",[]):
            html += f"<li><b>{m['name']}</b> • {m.get('dose','')} • {m.get('frequency','')} • {m.get('duration','')} — <span class='small'>{m.get('notes','')}</span></li>"
        html += "</ul>"
        tests = entry.get("tests",[])
        html += f"<p><b>Recommended Tests:</b> {', '.join(tests) if tests else 'As advised by physician'}</p>"
        em = entry.get("emergency_triggers",[])
        html += f"<p style='color:#b91c1c'><b>Emergency triggers (seek immediate care):</b> {', '.join(em)}</p>"
        html += f"<p><b>Lifestyle / Advice:</b> {entry.get('advice','Follow-up with physician')}</p>"
        html += "</div>"
        display(HTML(html))

ai_btn.on_click(ai_get_reco)
ai_clear.on_click(lambda _: out_ai.clear_output())

# ---------------------------
# Refresh all UI & init
# ---------------------------
def refresh_all():
    refresh_dropdowns()
    refresh_filters()
    apply_filters()
    render_records()
    show_transactions()
    show_booked()
    # update billing display label for first pid
    if not patients.empty:
        billing_pid.options = patients["PatientID"].tolist()
        billing_pid.value = patients["PatientID"].tolist()[0]

def init_ui():
    refresh_all()
    # set initial selections
    if not patients.empty:
        pid_dropdown.value = patients["PatientID"].tolist()[0]
        ud_select.value = patients["PatientID"].tolist()[0]
        billing_pid.value = patients["PatientID"].tolist()[0]
        appoint_patient.value = patients["PatientID"].tolist()[0]

init_ui()

# ---------------------------
# Build Tabs
# ---------------------------
# Patient Records tab
tab_records = widgets.VBox([widgets.HTML("<div class='card'><b>Patient Records</b></div>"), out_records])

# Add Patient tab
tab_add = widgets.VBox([
    widgets.HTML("<div class='card'><b>Add Patient</b></div>"),
    widgets.HBox([add_name, add_age, add_gender]),
    widgets.HBox([add_disease, add_doctor, add_base_fee]),
    widgets.HBox([add_tests, add_btn]),
    out_add
])

# View Patients tab
tab_view = widgets.VBox([
    widgets.HTML("<div class='card'><b>View & Filter Patients</b></div>"),
    widgets.HBox([search_text, apply_filters_btn]),
    widgets.HBox([disease_filter, condition_filter, gender_filter]),
    out_view
])

apply_filters_btn.on_click(apply_filters)
search_text.on_submit(lambda _: apply_filters())

# Search tab (quick)
search_btn = widgets.Button(description="Search", button_style="info")
def quick_search(_):
    with out_search:
        clear_output()
        q = (search_text.value or "").strip().lower()
        if not q:
            display(HTML("<b>Enter search term.</b>")); return
        df = patients[patients["Name"].fillna("").str.lower().str.contains(q) | patients["PatientID"].fillna("").str.lower().str.contains(q)]
        if df.empty:
            display(HTML("<b>No matches.</b>")); return
        df2 = df.copy(); df2["Tests"] = df2["Tests"].apply(lambda x:", ".join(x) if isinstance(x,(list,tuple)) else str(x))
        display(style_table(df2.reset_index(drop=True)))
search_btn.on_click(quick_search)

tab_search = widgets.VBox([widgets.HTML("<div class='card'><b>Search</b></div>"), widgets.HBox([search_text, search_btn]), out_search])

# Update/Delete tab
tab_update = widgets.VBox([
    widgets.HTML("<div class='card'><b>Update / Delete</b></div>"),
    widgets.HBox([ud_select, ud_refresh]),
    widgets.HBox([ud_name, ud_age, ud_gender]),
    widgets.HBox([ud_disease, ud_doctor]),
    widgets.HBox([ud_bill, ud_paid, ud_condition]),
    widgets.HBox([ud_tests]),
    widgets.HBox([ud_update, ud_delete]),
    out_update
])

# Billing tab
tab_billing = widgets.VBox([
    widgets.HTML("<div class='card'><b>Billing & Payment</b></div>"),
    widgets.HBox([billing_pid, billing_amount_display]),
    widgets.HBox([billing_custom_override, billing_calc_btn]),
    widgets.HBox([billing_half_btn, billing_full_btn, billing_msg]),
    out_billing,
    widgets.HTML("<div class='small'>Transactions:</div>"),
    widgets.HBox([view_tx_btn]),
    out_transactions
])

# Appointment tab
tab_appoint = widgets.VBox([
    widgets.HTML("<div class='card'><b>Appointments</b></div>"),
    widgets.HBox([appoint_patient, appoint_doctor, appoint_date]),
    widgets.HBox([appoint_time, appoint_reason, book_btn]),
    out_appointments
])

# AI Recommender tab
tab_ai = widgets.VBox([
    widgets.HTML("<div class='card'><b>AI Recommender (Professional)</b></div>"),
    widgets.HBox([ai_disease, ai_btn, ai_clear]),
    out_ai
])

# Assemble tabs in order requested
tabs = widgets.Tab()
tabs.children = [tab_records, tab_add, tab_view, tab_search, tab_update, tab_billing, tab_appoint, tab_ai]
titles = ["Patient Records","Add Patient","View Patients","Search","Update/Delete","Billing & Payment","Appointments","AI Recommender"]
for i,t in enumerate(titles):
    tabs.set_title(i,t)

display(tabs)

# Final message
print("✅ System initialized. Use the tabs to manage patients, billing, appointments and view AI recommendations.")

✅ System initialized. Use the tabs to manage patients, billing, appointments and view AI recommendations.
